In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display

PLL_DIR = Path.cwd().resolve().parents[2] / "results" / "pll"  # notebook lives in sw/tools/notebooks/
RESULT_CSV = None  # e.g. PLL_DIR / "pll_range_S5_20261013_150000.csv"; None = newest run

if RESULT_CSV is None:
    RESULT_CSV = sorted((p for p in PLL_DIR.glob("pll_range_*.csv") if not p.stem.endswith("_traces")),
                        key=lambda p: p.stat().st_mtime)[-1]
RESULT_CSV = Path(RESULT_CSV)
TRACE_CSV = RESULT_CSV.with_name(RESULT_CSV.stem + "_traces.csv")
runs = pd.read_csv(RESULT_CSV).sort_values("f_ref").reset_index(drop=True)
traces = pd.read_csv(TRACE_CSV) if TRACE_CSV.exists() else pd.DataFrame(columns=["f_ref", "kind"])

# Locked = lock bit set, carrier on the pad at f_vco / N (divider check) and the lock bit still set after the
# measurement. Everything else with the lock bit set (no carrier, wrong frequency, lost lock, error) is "check failed".
runs["good"] = (runs["status"] == "ok") & (runs["divider_ok"] == 1) & (runs["locked_end"] == 1)
runs["category"] = np.select([runs["status"] == "no_vco_setting", runs["status"] == "not_locked", runs["good"]],
                             ["no VCO setting", "not locked", "locked"], "lock bit set, check failed")
r0 = runs.iloc[0]
FILTER_TXT = f"Icp {r0.set_current:.0f} C1 {r0.set_c1:.0f} C2 {r0.set_c2:.0f} R1 {r0.set_r1:.0f}"

# Lock range: the longest stretch of neighbouring locked points; each edge lies between the last locked point and the
# next measured one (the sweep step).
good = np.r_[runs["good"].to_numpy(), False]
best, start = (0, 0), None
for i, ok in enumerate(good):
    if ok and start is None:
        start = i
    if not ok and start is not None:
        best = max(best, (start, i), key=lambda b: b[1] - b[0])
        start = None
if best[1] > best[0]:
    LOCK = (runs["f_ref"][best[0]], runs["f_ref"][best[1] - 1])
    below = runs["f_ref"][best[0] - 1] if best[0] > 0 else None
    above = runs["f_ref"][best[1]] if best[1] < len(runs) else None
    print(f"{RESULT_CSV.name}, loop filter {FILTER_TXT}: locks from ref {LOCK[0] / 1e6:.3f} to {LOCK[1] / 1e6:.3f} MHz "
          f"(VCO {LOCK[0] * 128 / 1e9:.3f} - {LOCK[1] * 128 / 1e9:.3f} GHz), {best[1] - best[0]} points")
    print("  lower edge between " + (f"{below / 1e6:.3f} and " if below else "below the sweep, at or under ")
          + f"{LOCK[0] / 1e6:.3f} MHz; upper edge between {LOCK[1] / 1e6:.3f}"
          + (f" and {above / 1e6:.3f} MHz" if above else " MHz and above the sweep"))
else:
    LOCK = None
    print(f"{RESULT_CSV.name}, loop filter {FILTER_TXT}: no locked point")
print(runs["category"].value_counts().to_string())

# Jitter-power figure of merit: 10 log10((jitter / 1 s)^2 x P / 1 mW), full-curve jitter over the common band
fom = 10 * np.log10((runs["time_jitter_trace_common_ps"] * 1e-12) ** 2 * runs["pll_p"] * 1e3)
display(pd.DataFrame({
    "ref [MHz]": runs["f_ref"] / 1e6,
    "VCO [MHz]": runs["f_vco"] / 1e6,
    "setting": [f"c{r.vco_tune_coarse:.0f} min {r.vco_current_min:.0f} max {r.vco_current_max:.0f}"
                if pd.notna(r.vco_tune_coarse) else "-" for r in runs.itertuples()],
    "VCO pick": runs["vco_pick"] if "vco_pick" in runs else "lut",  # "fastest": no table setting
    "Vctrl pred [V]": runs["vctrl_pred"],
    "|Kvco| pred [MHz/V]": runs["kvco_pred"] / 1e6,
    "result": runs["category"],
    "status": runs["status"],
    "VCO meas [MHz]": runs["f_vco_meas"] / 1e6,
    "carrier [dBm]": runs["carrier_dbm"],
    "P [mW]": runs["pll_p"] * 1e3,
    "jitter curve [ps]": runs["time_jitter_trace_ps"],
    "jitter noise [ps]": runs["time_jitter_noise_ps"],
    "jitter spurs [ps]": runs["time_jitter_spur_ps"],
    "jitter markers [ps]": runs["time_jitter_ps"],
    "FOM [dB]": fom,
}).round(3))

pll_range_S5_20261013_161640.csv, loop filter Icp 7 C1 0 C2 0 R1 7: locks from ref 7.079 to 19.953 MHz (VCO 0.906 - 2.554 GHz), 10 points
  lower edge between 6.310 and 7.079 MHz; upper edge between 19.953 and 22.387 MHz
category
locked                        21
not locked                    16
lock bit set, check failed     3
no VCO setting                 1


,ref [MHz],VCO [MHz],setting,VCO pick,Vctrl pred [V],|Kvco| pred [MHz/V],result,status,VCO meas [MHz],carrier [dBm],P [mW],jitter curve [ps],jitter noise [ps],jitter spurs [ps],jitter markers [ps],FOM [dB]
0,1.000,128.000,c14 min 15 max 14,lut,0.630,2023.319,not locked,not_locked,NaN,NaN,0.292,NaN,NaN,NaN,NaN,NaN
1,1.122,143.618,c14 min 15 max 14,lut,0.622,2023.319,not locked,not_locked,NaN,NaN,0.298,NaN,NaN,NaN,NaN,NaN
2,1.259,161.142,c14 min 15 max 14,lut,0.614,2023.319,not locked,not_locked,NaN,NaN,0.304,NaN,NaN,NaN,NaN,NaN
3,1.413,180.805,c14 min 15 max 14,lut,0.604,2023.319,not locked,not_locked,NaN,NaN,0.311,NaN,NaN,NaN,NaN,NaN
4,1.585,202.866,c12 min 14 max 14,lut,0.622,1975.537,not locked,not_locked,NaN,NaN,0.307,NaN,NaN,NaN,NaN,NaN
5,1.778,227.620,c12 min 14 max 14,lut,0.609,1975.537,not locked,not_locked,NaN,NaN,0.318,NaN,NaN,NaN,NaN,NaN
6,1.995,255.394,c7 min 14 max 15,lut,0.588,2000.235,not locked,not_locked,NaN,NaN,0.306,NaN,NaN,NaN,NaN,NaN
7,2.239,286.556,c7 min 14 max 15,lut,0.572,2000.235,not locked,not_locked,NaN,NaN,0.313,NaN,NaN,NaN,NaN,NaN
8,2.512,321.521,c7 min 14 max 15,lut,0.555,2000.235,not locked,not_locked,NaN,NaN,0.321,NaN,NaN,NaN,NaN,NaN
9,2.818,360.753,c14 min 13 max 6,lut,0.626,1993.291,not locked,not_locked,NaN,NaN,0.358,NaN,NaN,NaN,NaN,NaN


In [3]:
import sys

sys.path.insert(0, str(PLL_DIR.parents[1]))  # repo root
from plotly.colors import sample_colorscale
from plotly.subplots import make_subplots
from sw.lib.pll_util import bin_phase_noise

INK, GRID = "#52514e", "#e4e3df"
PALETTE = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7", "#e34948"]
CATEGORY_COLORS = {"locked": PALETTE[2], "lock bit set, check failed": PALETTE[3], "not locked": PALETTE[7],
                   "no VCO setting": "#a3a29e"}
OFFSET_AXIS = dict(type="log", tickvals=[1e2, 1e3, 1e4, 1e5, 1e6, 1e7, 1e8],
                   ticktext=["100 Hz", "1 kHz", "10 kHz", "100 kHz", "1 MHz", "10 MHz", "100 MHz"],
                   minor=dict(showgrid=True, gridcolor="#f1f0ed"))
N_BINS = 200  # points per plotted phase-noise curve; None = every measured point
COLORSCALE = "Viridis"  # curves coloured by the VCO frequency (log scale)

# Locked points, each with a colour on COLORSCALE by log10(f_vco) over the sweep
ok = runs[runs["good"]]
_lf = np.log10(runs["f_vco"])
COLOR_OF = dict(zip(runs["f_ref"], sample_colorscale(COLORSCALE, list((_lf - _lf.min()) / max(np.ptp(_lf), 1e-9)))))


def trace_of(f_ref, kind):
    return traces[(traces["f_ref"] == f_ref) & (traces["kind"] == kind)]


def in_range(df, f_start, f_stop):
    """Rows of df with the offset (column freq) between f_start and f_stop, so the y axis fits the shown band."""
    return df[(df["freq"] >= f_start) & (df["freq"] <= f_stop)]


def binned(df, f_start, f_stop):
    """The curve in df (columns freq, level_dbm) between f_start and f_stop, averaged into N_BINS log-spaced bins in
    linear power (pll_util.bin_phase_noise): display only, the jitter uses the measured points."""
    df = in_range(df, f_start, f_stop).sort_values("freq")
    if N_BINS is None or len(df) < 2:
        return df
    freq, level = bin_phase_noise(df["freq"], df["level_dbm"], np.geomspace(f_start, f_stop, N_BINS + 1))
    return pd.DataFrame({"freq": freq, "level_dbm": level})


def add_colorbar(fig):
    """Colour bar for the VCO frequency (invisible trace carrying the colour scale)."""
    fig.add_scatter(x=[None], y=[None], mode="markers", showlegend=False, hoverinfo="skip",
                    marker=dict(colorscale=COLORSCALE, cmin=_lf.min(), cmax=_lf.max(), color=[_lf.min()],
                                colorbar=dict(title="VCO", tickvals=np.log10([2e8, 5e8, 1e9, 2e9, 5e9, 1e10]),
                                              ticktext=["200 MHz", "500 MHz", "1 GHz", "2 GHz", "5 GHz", "10 GHz"])))


def save(fig, name):
    """Interactive HTML and a static PNG (kaleido) next to the CSV, then show the figure."""
    base = RESULT_CSV.with_name(f"{RESULT_CSV.stem}_{name}")
    fig.write_html(base.with_suffix(".html"))
    try:
        fig.write_image(base.with_suffix(".png"), width=1100, height=fig.layout.height or 600, scale=2)
        print(f"saved {base}.html and .png")
    except Exception as e:  # kaleido >= 1.0 needs a Chrome/Chromium install (`plotly_get_chrome`)
        print(f"saved {base}.html; PNG export failed: {e}")
    fig.show()

In [4]:
# Lock range overview against the reference frequency (VCO = 128 x ref): result per point, PLL supply power (every
# configured point), RMS jitter (locked points: full curve total, noise only, spurs, markers) and the lookup table's
# expected Vctrl of the chosen VCO setting. Shaded: the lock range (longest stretch of locked points).
fig = make_subplots(rows=4, cols=1, shared_xaxes=True, vertical_spacing=0.04, row_heights=[0.13, 0.29, 0.29, 0.29])
hover = "ref %{x:.3f} MHz (VCO %{customdata:.0f} MHz)"
for cat, color in CATEGORY_COLORS.items():
    d = runs[runs["category"] == cat]
    if d.empty:
        continue
    fig.add_scatter(x=d["f_ref"] / 1e6, y=[cat] * len(d), mode="markers", name=cat, row=1, col=1,
                    marker=dict(color=color, size=10, symbol="square"), customdata=d["f_vco"] / 1e6,
                    hovertemplate=f"{cat}<br>{hover}<extra></extra>")
    fig.add_scatter(x=d["f_ref"] / 1e6, y=d["pll_p"] * 1e3, mode="markers", showlegend=False, row=2, col=1,
                    marker=dict(color=color, size=8), customdata=d["f_vco"] / 1e6,
                    hovertemplate=f"%{{y:.3f}} mW<br>{hover}<extra></extra>")
    fig.add_scatter(x=d["f_ref"] / 1e6, y=d["vctrl_pred"], mode="markers", showlegend=False, row=4, col=1,
                    marker=dict(color=color, size=8), customdata=d["f_vco"] / 1e6,
                    hovertemplate=f"Vctrl %{{y:.3f}} V<br>{hover}<extra></extra>")
for col, name, color, dash in (("time_jitter_trace_ps", "jitter: full curve, total", PALETTE[0], "solid"),
                               ("time_jitter_noise_ps", "jitter: noise only", PALETTE[2], "solid"),
                               ("time_jitter_spur_ps", "jitter: spurs", PALETTE[3], "dot"),
                               ("time_jitter_ps", "jitter: markers", PALETTE[1], "dash")):
    fig.add_scatter(x=ok["f_ref"] / 1e6, y=ok[col], mode="lines+markers", name=name, row=3, col=1,
                    line=dict(color=color, width=2, dash=dash), marker=dict(size=7), customdata=ok["f_vco"] / 1e6,
                    hovertemplate=f"{name} %{{y:.2f}} ps<br>{hover}<extra></extra>")
if LOCK:
    fig.add_vrect(x0=LOCK[0] / 1e6, x1=LOCK[1] / 1e6, fillcolor=PALETTE[2], opacity=0.08, line_width=0)
fig.update_layout(title=f"Lock range, loop filter {FILTER_TXT}<br><sub>{RESULT_CSV.name}</sub>",
                  template="plotly_white", height=900, legend=dict(orientation="h", y=-0.08),
                  hoverlabel=dict(namelength=-1))
fig.update_xaxes(type="log", gridcolor=GRID)
fig.update_xaxes(title="reference frequency [MHz] (VCO = 128 x ref)", row=4)
fig.update_yaxes(gridcolor=GRID)
fig.update_yaxes(showticklabels=False, row=1)
fig.update_yaxes(title="power [mW]", rangemode="tozero", row=2)
fig.update_yaxes(title="RMS jitter [ps]", type="log", row=3)
fig.update_yaxes(title="Vctrl pred [V]", row=4)
save(fig, "overview")

saved /users/micas/wvandest/projects/lagd-meas/results/pll/pll_range_S5_20261013_161640_overview.html and .png


In [5]:
F_start, F_stop = 1e3, 10e6  # offset range shown [Hz]

# Phase noise referred to the VCO (full curve on the pad + 20 log10 N) of every locked point, coloured by the VCO
# frequency. Binned to N_BINS points (linear power average, display only).
fig = go.Figure()
for r in ok.itertuples():
    d = binned(trace_of(r.f_ref, "L_trace"), F_start, F_stop)
    fig.add_scatter(x=d["freq"], y=d["level_dbm"] + 20 * np.log10(r.total_div), mode="lines", showlegend=False,
                    line=dict(color=COLOR_OF[r.f_ref], width=1.5),
                    hovertemplate=f"VCO {r.f_vco / 1e6:.0f} MHz (ref {r.f_ref / 1e6:.3f} MHz)<br>"
                                  "%{x:.3s}Hz: %{y:.1f} dBc/Hz<extra></extra>")
add_colorbar(fig)
fig.update_layout(title=f"Phase noise referred to the VCO, locked points, loop filter {FILTER_TXT}",
                  template="plotly_white", height=550)
fig.update_xaxes(title="offset from the carrier", gridcolor=GRID, **OFFSET_AXIS,
                 range=[np.log10(F_start), np.log10(F_stop)])
fig.update_yaxes(title="L(f) referred to the VCO [dBc/Hz]", gridcolor=GRID)
save(fig, "phase_noise")

saved /users/micas/wvandest/projects/lagd-meas/results/pll/pll_range_S5_20261013_161640_phase_noise.html and .png


In [ ]:
F_start, F_stop = None, None  # offset from the carrier shown [Hz] (may be negative); None = edge of the traces

# Averaged spectrum on the pad around the carrier of every locked point, relative to the carrier (trace maximum), in
# the trace's RBW (not binned), coloured by the VCO frequency.
fig = go.Figure()
for r in ok.itertuples():
    d = trace_of(r.f_ref, "spectrum")
    if d.empty:
        continue
    d = d.assign(freq=d["freq"] - d["center"], level_dbc=d["level_dbm"] - d["level_dbm"].max())
    d = in_range(d, d["freq"].min() if F_start is None else F_start, d["freq"].max() if F_stop is None else F_stop)
    fig.add_scatter(x=d["freq"], y=d["level_dbc"], mode="lines", showlegend=False,
                    line=dict(color=COLOR_OF[r.f_ref], width=1),
                    hovertemplate=f"VCO {r.f_vco / 1e6:.0f} MHz, pad {r.f_out_meas / 1e6:.3f} MHz, "
                                  f"carrier {r.carrier_dbm:.1f} dBm<br>%{{x:.4s}}Hz: %{{y:.1f}} dBc<extra></extra>")
add_colorbar(fig)
rbw = traces.loc[traces["kind"] == "spectrum", "rbw"]
fig.update_layout(title=f"Spectrum on the pad around the carrier, locked points"
                        + (f" (RBW {rbw.iloc[0]:g} Hz)" if len(rbw) else "") + f", loop filter {FILTER_TXT}",
                  template="plotly_white", height=550)
fig.update_xaxes(title="offset from the carrier", type="linear", ticksuffix="Hz", exponentformat="SI", gridcolor=GRID)
fig.update_yaxes(title="level relative to the carrier [dBc in RBW]", gridcolor=GRID)
save(fig, "spectrum")

saved /users/micas/wvandest/projects/lagd-meas/results/pll/pll_range_S5_20261013_161640_spectrum.html and .png


: 